**Instalación de PySpark**

In [ ]:
!pip install pyspark

**Importación de la librería SparkSession**

In [ ]:
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("DatosSaludJunin") \
    .getOrCreate()

print("Spark iniciado correctamente")

Spark iniciado correctamente


**Cargar el archivo como DataFrame de PySpark**

In [ ]:
df = spark.read.csv(
    "Dataset_Salud_BIGDATA.csv",
    header=True,
    sep=";",
    inferSchema=True
)

df.show(5)

+--------+----------------+--------------+----------+--------------+-------------+--------------+-------------+--------------+-------------+--------------+-------------+--------------+-------------+-----------+-----------+-----------+
|  cod_pc|fecha_nacimiento|Id_Financiador|valor_labo|fecha_dosaje_1|hemoglobina_1|fecha_dosaje_2|hemoglobina_2|fecha_dosaje_3|hemoglobina_3|fecha_dosaje_4|hemoglobina_4|fecha_dosaje_5|hemoglobina_5|fecha_supT1|fecha_supT2|fecha_supT3|
+--------+----------------+--------------+----------+--------------+-------------+--------------+-------------+--------------+-------------+--------------+-------------+--------------+-------------+-----------+-----------+-----------+
|*****728|       1/08/1998|             2|       LEV|    30/12/2024|         12.5|    29/01/2025|         12.6|          NULL|         NULL|          NULL|         NULL|          NULL|         NULL| 30/12/2024| 29/01/2025| 28/02/2025|
|*****636|      15/08/1999|             2|       MOD|    27/

**Filas y columnas**

In [ ]:
print("Filas:", df.count())
print("Columnas:", len(df.columns))

Filas: 4791
Columnas: 17


**Estructura**

In [ ]:
df.printSchema()

root
 |-- cod_pc: string (nullable = true)
 |-- fecha_nacimiento: string (nullable = true)
 |-- Id_Financiador: integer (nullable = true)
 |-- valor_labo: string (nullable = true)
 |-- fecha_dosaje_1: string (nullable = true)
 |-- hemoglobina_1: string (nullable = true)
 |-- fecha_dosaje_2: string (nullable = true)
 |-- hemoglobina_2: string (nullable = true)
 |-- fecha_dosaje_3: string (nullable = true)
 |-- hemoglobina_3: string (nullable = true)
 |-- fecha_dosaje_4: string (nullable = true)
 |-- hemoglobina_4: string (nullable = true)
 |-- fecha_dosaje_5: string (nullable = true)
 |-- hemoglobina_5: string (nullable = true)
 |-- fecha_supT1: string (nullable = true)
 |-- fecha_supT2: string (nullable = true)
 |-- fecha_supT3: string (nullable = true)



**Conteo de registro según el valor_labo**

In [ ]:
from pyspark.sql.functions import count

resultado_df = df.groupBy("valor_labo") \
    .agg(count("*").alias("Cantidad")) \
    .orderBy("Cantidad", ascending=False)

resultado_df.show()

+----------+--------+
|valor_labo|Cantidad|
+----------+--------+
|       LEV|    2951|
|       MOD|    1399|
|      NULL|     399|
|       SEV|      39|
|       lev|       2|
|         2|       1|
+----------+--------+



**Medición del tiempo de ejecución**

In [ ]:
inicio = time.time()
resultado_df = df.groupBy("valor_labo") \
    .agg(count("*").alias("Cantidad")) \
    .orderBy("Cantidad", ascending=False)
resultado_df.collect()
fin = time.time()
tiempo_pyspark_df = fin - inicio
print("Tiempo PySpark DataFrame:", tiempo_pyspark_df, "segundos")

Tiempo PySpark DataFrame: 1.9330217838287354 segundos


**Creación de una vista temporal**

In [ ]:
df.createOrReplaceTempView("gestantes")

**Consulta ejecutada con Spark SQL**

In [ ]:
inicio = time.time()
resultado_sql = spark.sql("""
    SELECT
        valor_labo,
        COUNT(*) AS Cantidad
    FROM gestantes
    GROUP BY valor_labo
    ORDER BY Cantidad DESC
""")
fin = time.time()
tiempo_spark_sql = fin - inicio
print("Tiempo Spark SQL:", tiempo_spark_sql, "segundos")

Tiempo Spark SQL: 0.032224178314208984 segundos


**Promedio de la consulta hecha en SQL Server**

In [ ]:
tiempos_sql = [4, 6, 7, 9, 4, 4, 9]

promedio_sql = sum(tiempos_sql) / len(tiempos_sql)

print("Promedio SQL Server:", promedio_sql, "ms")
print("Promedio SQL Server:", promedio_sql / 1000, "segundos")

Promedio SQL Server: 6.142857142857143 ms
Promedio SQL Server: 0.0061428571428571435 segundos
